# Lecture 1 · Notebook 2 — What is a language model?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture01/lecture01_2_what_is_a_language_model.ipynb)

**Companion to [Chapter 7 of the video lecture](https://nlp-course-cuhksz.github.io/lecture01.html#p7-s77)** (slides 76–82). A language model assigns a probability to a sequence of
words, or, equivalently, predicts the **next word given the previous ones**:
$p(w_n \mid w_1, \dots, w_{n-1})$. Slide 80 calls this conditional probability *the foundation of modern large language models*.
In this notebook you will:

1. build a **bigram** language model from counts and see why it breaks on unseen word pairs,
2. score sentences with **GPT-2**: the chimpanzee's *"Sfklkljf fskjhfkjsh"* vs. *"ChatGPT is all you need"* (slide 78),
3. break *"the cat sat on the mat"* into a **product of conditionals** with the chain rule (slide 82),
4. get **next-word suggestions** like a phone keyboard (slide 79), and
5. write the **generation loop** yourself: predict, pick, append, repeat (slide 80).

Runs on CPU in about three minutes (GPT-2 small, 124M parameters, ~550 MB download).

In [ ]:
!pip -q install -U transformers

## 1 · A bigram model from counts
The oldest language models (slide 76, the n-gram model going back to Shannon) estimate
$p(w_n \mid w_{n-1}) = \dfrac{\text{count}(w_{n-1}, w_n)}{\text{count}(w_{n-1})}$ from a corpus. Our corpus is twelve sentences.

In [ ]:
import math
from collections import Counter, defaultdict
corpus = '''the cat sat on the mat . the dog sat on the rug . the cat ate the fish .
the dog ate the bone . a cat sat on a chair . the bird sat on the tree .
the cat saw the dog . the dog saw the cat . a dog sat on the mat .
the fish swam in the sea . the bird ate the seed . the cat sat on the rug .'''
sents = [["<s>"] + s.split() + ["</s>"] for s in corpus.replace("\n", " ").split(" . ") if s.strip()]
sents = [[w for w in s if w != "."] for s in sents]
uni, bi = Counter(), Counter()
for s in sents:
    uni.update(s[:-1]); bi.update(zip(s[:-1], s[1:]))
V = sorted(set(w for s in sents for w in s))
print(len(sents), "sentences,", len(V), "word types")

def p_bigram(prev, w, k=0.0):                 # k > 0 gives add-k smoothing
    return (bi[(prev, w)] + k) / (uni[prev] + k * len(V))

def score(sentence, k=0.0):
    w = ["<s>"] + sentence.split() + ["</s>"]
    logp = 0.0
    for a, b in zip(w[:-1], w[1:]):
        p = p_bigram(a, b, k)
        if p == 0: return float("-inf")
        logp += math.log(p)
    return logp

print("next-word distribution after 'the':",
      {w: round(p_bigram("the", w), 2) for w in V if p_bigram("the", w) > 0})
for s in ["the cat sat on the mat", "the dog sat on the chair", "mat the on sat cat the"]:
    print(f"{s:28s} log p = {score(s):8.3f}   (add-1 smoothed: {score(s, k=1):.3f})")

*"the dog sat on the chair"* is a perfectly good sentence, but the pair *(the, chair)* never occurs in the corpus, so the unsmoothed model gives
it probability **zero**. Smoothing patches this, but counts cannot tell that *chair* is similar to *mat* and *rug*. Neural language models
(slide 81) solve this by representing words as vectors, so similar words share statistics.

## 2 · Scoring sentences with GPT-2
GPT-2 computes the same kind of conditional probabilities, but with a neural network over **tokens** (pieces of words).

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.manual_seed(0)
tok = AutoTokenizer.from_pretrained("gpt2")
lm = AutoModelForCausalLM.from_pretrained("gpt2").eval()
print(f"GPT-2 small: {sum(p.numel() for p in lm.parameters()) / 1e6:.0f}M parameters, vocabulary {tok.vocab_size}")

@torch.no_grad()
def token_logprobs(text):
    ids = tok(tok.bos_token + text, return_tensors="pt").input_ids        # start from <|endoftext|>
    logp = F.log_softmax(lm(ids).logits[0, :-1], dim=-1)
    lp = logp.gather(1, ids[0, 1:, None])[:, 0]
    return [tok.decode(i) for i in ids[0, 1:]], lp

for text in ["Sfklkljf fskjhfkjsh", "ChatGPT is all you need.", "need you all is ChatGPT."]:
    toks, lp = token_logprobs(text)
    print(f"{text:28s} {len(toks):2d} tokens   log p = {lp.sum():7.2f}   per token = {lp.mean():6.2f}")

The chimpanzee's string is cut into many pieces and gets by far the lowest **total** log-probability. (Per token it is not much worse:
once the model has seen a few random letters, more random letters are less surprising. And *ChatGPT* itself is an unusual word for
GPT-2, which was trained in 2019, before ChatGPT existed.) Shuffling the words of the fluent sentence costs a lot, with exactly the same
tokens: the model has learned spelling, grammar and common phrases just by predicting the next token.

## 3 · The chain rule: a sentence is a product of conditionals
Slide 82: $P(\text{the cat sat on the mat}) = P(\text{the}) \cdot P(\text{cat} \mid \text{the}) \cdot P(\text{sat} \mid \text{the cat}) \cdots$
Here are GPT-2's actual factors.

In [ ]:
toks, lp = token_logprobs("the cat sat on the mat")
history = ""
for t, l in zip(toks, lp):
    print(f"P({t!r:8s} | {history!r:22s}) = {l.exp():.4f}")
    history += t
print(f"product = {lp.sum().exp():.3e}   (log p = {lp.sum():.2f})")

# slide 79: P(next | history) = P(history + next) / P(history)
_, lp_full = token_logprobs("the cat sat on the mat")
_, lp_hist = token_logprobs("the cat sat on the")
print(f"P(full) / P(history) = {(lp_full.sum() - lp_hist.sum()).exp():.4f}  = the last factor above")

## 4 · Next-word suggestions, like a phone keyboard
Slide 79: type *"I've"* and the keyboard suggests *been*, *got*, *never*. That is a conditional language model showing its top guesses.

In [ ]:
@torch.no_grad()
def next_words(prefix, k=8):
    ids = tok(prefix, return_tensors="pt").input_ids
    p = lm(ids).logits[0, -1].softmax(-1)
    top = p.topk(k)
    return [(tok.decode(i), round(v.item(), 3)) for v, i in zip(top.values, top.indices)]

for prefix in ["I've", "The capital of France is", "Natural language processing is"]:
    print(f"{prefix!r:36s} -> {next_words(prefix)}")

Notice *"The capital of France is"*: *Paris* is only one guess among many. GPT-2 predicts plausible web text, not facts; a
continuation such as *"the city of Paris"* is just as likely. Knowledge and instruction following get stronger with scale and post-training.

## 5 · Generation is next-token prediction in a loop
This is everything ChatGPT does at inference time (slide 80): compute the distribution over the next token, pick one, append it, repeat.
Greedy decoding always picks the most likely token; sampling draws from the distribution, and the **temperature** sharpens or flattens it.

In [ ]:
@torch.no_grad()
def generate(prompt, steps=25, temperature=0.0):
    ids = tok(prompt, return_tensors="pt").input_ids
    for _ in range(steps):
        logits = lm(ids).logits[0, -1]
        if temperature == 0:
            nxt = logits.argmax()                                          # greedy
        else:
            nxt = torch.multinomial((logits / temperature).softmax(-1), 1)[0]   # sampling
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return tok.decode(ids[0])

prompt = "In the future, language models will"
print("greedy     :", generate(prompt))
for T in [0.7, 1.5]:
    print(f"T = {T:<8}:", generate(prompt, temperature=T))

GPT-2 is small (124M parameters) and was trained only to continue web text, so it rambles and repeats. GPT-3 uses the same objective with
175B parameters (slide 81); chat assistants add instruction tuning and RL on top (Lectures 5–6). Notebook 3 uses such an instruction-tuned model.

## Exercises
1. In section 2, try 我思故我在 (slide 81). How many tokens does GPT-2 need, and why does an English-only tokenizer handle Chinese so badly?
2. Change `"gpt2"` to `"distilgpt2"` (82M parameters) and compare the log-probabilities in section 2. Does the smaller model separate fluent and
   shuffled text as clearly?
3. In section 5, sample five times at T = 0.7 and at T = 1.5. Describe in one sentence what the temperature does to the output.

**Next:** [Notebook 3 — one model, many tasks]({colab(N3)})